In [3]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [ ]:
# 1. LOAD THE REPETITION DATASET
DATA_PATH = "../edited_datasets/squat_rep_vectors_ml.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Could not find {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

# 2. SEPARATE LABELS, GROUPS (VIDEOS), AND METRIC FEATURES
# Convert target labels to binary: good = 1, bad = 0
y = df['label'].map({'good': 1, 'bad': 0}).values
groups = df['video_name'].values

# Drop structural/metadata columns to leave pure kinematic metrics
X = df.drop(columns=['video_name', 'rep_number', 'label', 'body_side'])
feature_names = X.columns.tolist()

print(f"Dataset Loaded: {X.shape[0]} repetitions, {X.shape[1]} features.")
print(f"Class Breakdown -> Good Form: {np.sum(y==1)}, Bad Form: {np.sum(y==0)}")

# 3. LEAK-PROOF EVALUATION VIA GROUPKFOLD
# We split by 3 groups because we have 9 unique videos (keeps 1-2 videos out per fold)
gkf = GroupKFold(n_splits=3)

oof_preds = np.zeros(len(df)) # Out-of-fold predictions
oof_probs = np.zeros(len(df)) # Out-of-fold probabilities

# Tightly regularized Random Forest to prevent overfitting on 35 samples
model_template = RandomForestClassifier(
    n_estimators=50, 
    max_depth=3, 
    random_state=42, 
    class_weight="balanced"
)

print("\n--- Starting Leak-Proof Cross Validation ---")
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]
    
    # Clone and train model clone for this specific fold
    fold_model = RandomForestClassifier(
        n_estimators=50, max_depth=3, random_state=42, class_weight="balanced"
    )
    fold_model.fit(X_train, y_train)
    
    oof_preds[val_idx] = fold_model.predict(X_val)
    oof_probs[val_idx] = fold_model.predict_proba(X_val)[:, 1]
    
    fold_acc = np.mean(oof_preds[val_idx] == y_val)
    print(f"Fold {fold+1} complete. Validation Accuracy: {fold_acc:.2f}")

# 4. REPORT CLASSIFICATION PERFORMANCE
print("\n=== OVERALL OUT-OF-FOLD PERFORMANCE ===")
print(classification_report(y, oof_preds, target_names=['bad', 'good']))
print("Confusion Matrix:")
print(confusion_matrix(y, oof_preds))
print(f"ROC-AUC Score: {roc_auc_score(y, oof_probs):.4f}")

# 5. RETRAIN FINAL MODEL ON ALL DATA & EXTRACT DIAGNOSTICS
print("\n--- Training Production Model ---")
final_model = RandomForestClassifier(
    n_estimators=50, max_depth=3, random_state=42, class_weight="balanced"
)
final_model.fit(X, y)

# Extract and rank the features that define form failure
importances = final_model.feature_importances_
indices = np.argsort(importances)[::-1]

print("\nTop Features Used to Classify Squat Form:")
for i in range(min(5, len(indices))):
    print(f"{i+1}. {feature_names[indices[i]]} (Importance: {importances[indices[i]]:.4f})")

# 6. SAVE ARTIFACT FOR DEPLOYMENT/INFERENCE
OUTPUT_PATH = "../models/"
MODEL_OUTPUT = "squat_classifier_model.pkl"

os.makedirs(OUTPUT_PATH, exist_ok=True)
joblib.dump(final_model, OUTPUT_PATH + MODEL_OUTPUT)
print(f"\nSuccess! Final model saved to '{MODEL_OUTPUT}'")

Dataset Loaded: 35 repetitions, 31 features.
Class Breakdown -> Good Form: 21, Bad Form: 14

--- Starting Leak-Proof Cross Validation ---
Fold 1 complete. Validation Accuracy: 0.91
Fold 2 complete. Validation Accuracy: 0.33
Fold 3 complete. Validation Accuracy: 0.92

=== OVERALL OUT-OF-FOLD PERFORMANCE ===
              precision    recall  f1-score   support

         bad       0.59      0.93      0.72        14
        good       0.92      0.57      0.71        21

    accuracy                           0.71        35
   macro avg       0.76      0.75      0.71        35
weighted avg       0.79      0.71      0.71        35

Confusion Matrix:
[[13  1]
 [ 9 12]]
ROC-AUC Score: 0.9354

--- Training Production Model ---

Top Features Used to Classify Squat Form:
1. sticking_point_velocity_drop (Importance: 0.1856)
2. peak_ascent_velocity_y (Importance: 0.1183)
3. torso_lean_to_depth_ratio (Importance: 0.0826)
4. avg_ascent_velocity_y (Importance: 0.0803)
5. max_torso_lean_degrees (Impor

In [10]:
def analyze_new_repetition(rep_vector_dict):
    """
    Receives a single repetition's 31 features as a dictionary,
    classifies it, and provides explicit biomechanical diagnostics.
    """
    # Load your trained model
    model = joblib.load("../models/squat_classifier_model.pkl")
    
    # Convert input to DataFrame matching the model's feature layout
    rep_df = pd.DataFrame([rep_vector_dict])
    
    # Run ML prediction (1 = good, 0 = bad)
    prediction = model.predict(rep_df)[0]
    
    feedback = []
    
    if prediction == 1:
        status = "Good Form"
        feedback.append("Great execution! Keep maintaining this depth and torso alignment.")
    else:
        status = "Bad Form Detected"
        
        # Diagnostic Check 1: Squat Depth
        # Lower depth ratios indicate a shallow squat
        if rep_vector_dict.get('max_depth_ratio', 1.0) < 0.25:
            feedback.append("• Problem: Shallow depth. Ensure your hips drop lower (hip crease below parallel with knees).")
            
        # Diagnostic Check 2: Torso Lean / Back Angle
        if rep_vector_dict.get('max_torso_lean_degrees', 0) > 60.0:
            feedback.append("• Problem: Excessive torso lean. Your chest is dropping forward too much, putting strain on your lower back.")
            
        # Diagnostic Check 3: Sticking Point / Velocity loss
        if rep_vector_dict.get('sticking_point_velocity_drop', 0) > 0.003:
            feedback.append("• Problem: Severe velocity drop on ascent. Work on explosive power out of the bottom position.")
            
        if not feedback:
            feedback.append("• Problem: Structural instability during ascent/descent phases.")

    return {
        "status": status,
        "diagnostics": feedback
    }

In [ ]:
single_rep_data = {
    'min_knee_angle': 42.98,               
    'min_hip_angle': 14.99,            
    'max_torso_lean_degrees': 68.21,          
    'max_depth_ratio': 0.196,                 
    'hip_to_ankle_bottom_alignment': 0.117,    
    'avg_knee_angle_descent': 84.44,
    'avg_hip_angle_descent': 53.83,
    'max_shin_forward_angle': 37.69,

    'descent_duration_seconds': 3.70,
    'ascent_duration_seconds': 2.60,
    'bottom_pause_duration_seconds': 0.36,
    'descent_to_ascent_ratio': 1.42,
    'total_rep_duration': 6.26,
    'time_to_max_jerk': 0.91,

    'shoulder_x_sd': 0.011,                
    'hip_x_sd': 0.031,                       
    'knee_x_sd': 0.022,                       
    'shoulder_hip_x_corr': -0.45,
    'hip_knee_x_corr': -0.79,
    'shoulder_max_x_drift': 0.051,
    'hip_max_x_drift': 0.112,
    'torso_lean_to_depth_ratio': 1.00,
    'knee_travel_to_depth_ratio': 0.37,
    'ascent_path_symmetry': 0.96,

    'avg_descent_velocity_y': 0.0015,
    'avg_ascent_velocity_y': 0.0020,
    'max_acceleration_y': 0.0008,
    'max_jerk_y': 0.0009,
    'peak_ascent_velocity_y': 0.0044,
    'sticking_point_velocity_drop': 0.0042,
    'velocity_loss_vs_rep_1': 1.00
}

In [11]:
# Pass it to your diagnostic function
result = analyze_new_repetition(single_rep_data)

# Access the results
print(result["status"])        # E.g., "Bad Form Detected"
print(result["diagnostics"])   # E.g., ["• Problem: Excessive torso lean...", "• Problem: Shallow depth..."]

Bad Form Detected
['• Problem: Shallow depth. Ensure your hips drop lower (hip crease below parallel with knees).', '• Problem: Excessive torso lean. Your chest is dropping forward too much, putting strain on your lower back.', '• Problem: Severe velocity drop on ascent. Work on explosive power out of the bottom position.']
